In [1]:
%%writefile /kaggle/working/03c_preprocess_mri_v3.py
import os
import numpy as np
import pandas as pd
import pydicom
import cv2

BASE    = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
SPLIT   = "test"
OUT_DIR = "/tmp/vol_test_v3"
IMG     = 224
DEPTH   = 32
VIEWS   = [("Sagittal", "fs"), ("Coronal", "fs"), ("Axial", "fs"),
           ("Sagittal", "nonfs"), ("Coronal", "nonfs")]


def choose_series(g, series_root):
    g = g.copy()
    g["n"] = [len(os.listdir(os.path.join(series_root, s)))
              if os.path.isdir(os.path.join(series_root, s)) else 0
              for s in g["SeriesInstanceUID"]]
    g = g[g["n"] > 0]
    chosen, used = [], set()
    for plane, kind in VIEWS:
        c = g[(g["Anatomical_Plane"] == plane) & ~g["SeriesInstanceUID"].isin(used)].copy()
        if kind == "fs":
            c["score"] = c["Fluid_Sensitive"] * 2 + c["Fat_Suppression"]
        else:
            c = c[c["Fat_Suppression"] == 0].copy()
            c["score"] = 1 - c["Fluid_Sensitive"]
        c = c.sort_values(["score", "n"], ascending=False)
        s = c["SeriesInstanceUID"].iloc[0] if len(c) else None
        chosen.append(s)
        if s is not None:
            used.add(s)
    return chosen


def load_series(path):
    slices = []
    for f in os.listdir(path):
        if not f.endswith(".dcm"):
            continue
        ds = pydicom.dcmread(os.path.join(path, f))
        try:
            pos = np.array(ds.ImagePositionPatient, dtype=float)
            ori = np.array(ds.ImageOrientationPatient, dtype=float)
            key = float(pos @ np.cross(ori[:3], ori[3:]))
        except Exception:
            key = float(getattr(ds, "InstanceNumber", 0))
        img = ds.pixel_array.astype(np.float32)
        if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
            img = img.max() - img
        slices.append((key, img))
    slices.sort(key=lambda x: x[0])
    return [s[1] for s in slices]


def square_resize(img, size):
    h, w = img.shape
    m = max(h, w)
    canvas = np.zeros((m, m), np.float32)
    y, x = (m - h) // 2, (m - w) // 2
    canvas[y:y + h, x:x + w] = img
    return cv2.resize(canvas, (size, size), interpolation=cv2.INTER_AREA)


def to_volume(slices):
    vol = np.stack([square_resize(s, IMG) for s in slices])
    fg = vol[vol > 0]
    lo, hi = np.percentile(fg if fg.size else vol, [0.5, 99.5])
    vol = np.clip((vol - lo) / (hi - lo + 1e-6), 0, 1)
    idx = np.linspace(0, len(vol) - 1, DEPTH).round().astype(int)
    return (vol[idx] * 255).astype(np.uint8)


def process_study(study_uid, g):
    series_root = os.path.join(BASE, f"{SPLIT}_series", study_uid)
    chosen = choose_series(g, series_root)
    vol = np.zeros((len(VIEWS), DEPTH, IMG, IMG), np.uint8)
    mask = np.zeros(len(VIEWS), np.uint8)
    for i, s_uid in enumerate(chosen):
        if s_uid is None:
            continue
        try:
            slices = load_series(os.path.join(series_root, s_uid))
            if slices:
                vol[i] = to_volume(slices)
                mask[i] = 1
        except Exception:
            pass
    np.savez_compressed(os.path.join(OUT_DIR, f"{study_uid}.npz"), vol=vol, mask=mask)

Writing /kaggle/working/03c_preprocess_mri_v3.py


In [2]:
import os, gc, glob, importlib.util
import numpy as np, pandas as pd, torch, pydicom
from tqdm.auto import tqdm

COMP = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
VOL = {"v1": "/tmp/vol_test", "v2": "/tmp/vol_test_v2", "v3": "/tmp/vol_test_v3"}

FILES = {}
for root, dirs, files in os.walk("/kaggle/input"):
    dirs[:] = [d for d in dirs if d != "competitions"]
    for f in files:
        if not f.endswith(".npz"):
            FILES.setdefault(f, os.path.join(root, f))

def find_file(name):
    if name not in FILES:
        raise FileNotFoundError(name)
    return FILES[name]

def load_module(name, path):
    print(f"{name:4s} -> {path}")
    spec = importlib.util.spec_from_file_location(name, path)
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
    return m

# nama: (backbone, file bobot, versi data, pakai normalisasi kiri-kanan?)
DINO = "vit_small_patch14_dinov2.lvd142m"
MODELS = {
    "effb0":                      ("efficientnet_b0", "efficientnet_b0_best.pt",            "v1", False),
    "dinov2s_student_v1d":        (DINO, "dinov2s_student_v1d_best.pt",                     "v1", False),
    "dinov2s_student_v1d_lat":    (DINO, "dinov2s_student_v1d_lat_best.pt",                 "v1", True),
    "dinov2s_student_v1d_lat_s7": (DINO, "dinov2s_student_v1d_lat_s7_best.pt",              "v1", True),
    "dinov2s_student_v2":         (DINO, "dinov2s_student_v2_best.pt",                      "v2", False),
    "effb0_student_v3":           ("efficientnet_b0", "effb0_student_v3_best.pt",           "v2", False),
    "dinov2s_student_d32_lat":    (DINO, "dinov2s_student_d32_lat_best.pt",                 "v3", True),
    "effb0_student_d32_s7":       ("efficientnet_b0", "effb0_student_d32_s7_best.pt",       "v3", False),
}
USE = list(MODELS)            # val 0.8839 | gold 0.8733

need = sorted({MODELS[n][2] for n in USE})
PP = {"v1": load_module("pp1", find_file("03_preprocess_mri.py")),
      "v2": load_module("pp2", find_file("03b_preprocess_mri_v2.py")),
      "v3": load_module("pp3", "/kaggle/working/03c_preprocess_mri_v3.py")}
tb = load_module("tb", find_file("11_train_student_lat.py"))      # KneeNet & KneeDS (mendukung cermin)
LABELS = tb.LABELS

for name in USE:
    print(f"{name:28s} lat={MODELS[name][3]!s:5s} {MODELS[name][1]:38s} -> {find_file(MODELS[name][1])}")
print("Data yang perlu diproses:", need, "| GPU:", torch.cuda.device_count())

pp1  -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-02-preprocess-mri/03_preprocess_mri.py
pp2  -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-02b-preprocess-mri-v2/03b_preprocess_mri_v2.py
pp3  -> /kaggle/working/03c_preprocess_mri_v3.py
tb   -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-24-lat-v2-d32/11_train_student_lat.py
effb0                        lat=False efficientnet_b0_best.pt                -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-03-train-baseline/efficientnet_b0_best.pt
dinov2s_student_v1d          lat=False dinov2s_student_v1d_best.pt            -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-16-student-diverse/dinov2s_student_v1d_best.pt
dinov2s_student_v1d_lat      lat=True  dinov2s_student_v1d_lat_best.pt        -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-23-dinov2-lat/dinov2s_student_v1d_lat_best.pt
dinov2s_student_v1d_lat_s7   lat=True  dinov2s_student_v1d_lat_s7_best.pt     -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-23-dinov

In [3]:
series = pd.read_csv(f"{COMP}/test_series.csv")
test = pd.read_csv(f"{COMP}/test.csv")
uids = test["StudyInstanceUID"].tolist()

avail = {}
for ver in need:
    pp = PP[ver]
    pp.SPLIT, pp.OUT_DIR = "test", VOL[ver]
    os.makedirs(VOL[ver], exist_ok=True)
    failed = 0
    for uid, g in tqdm(series.groupby("StudyInstanceUID"), desc=f"Preprocess {ver}"):
        try:
            pp.process_study(uid, g)
        except Exception as e:
            failed += 1
            print("Gagal:", ver, uid, type(e).__name__, e)
    avail[ver] = [u for u in uids if os.path.exists(f"{VOL[ver]}/{u}.npz")]
    print(f"{ver}: volume tersedia {len(avail[ver])}/{len(uids)} | gagal {failed}")
    assert len(avail[ver]) >= 0.95 * len(uids), f"Terlalu banyak studi gagal ({ver}), jangan disubmit"

Preprocess v1:   0%|          | 0/3 [00:00<?, ?it/s]

v1: volume tersedia 3/3 | gagal 0


Preprocess v2:   0%|          | 0/3 [00:00<?, ?it/s]

v2: volume tersedia 3/3 | gagal 0


Preprocess v3:   0%|          | 0/3 [00:00<?, ?it/s]

v3: volume tersedia 3/3 | gagal 0


In [4]:
def center_x(path):
    f = sorted(glob.glob(path + "/*.dcm"))
    if not f:
        return np.nan, ""
    d = pydicom.dcmread(f[0], stop_before_pixels=True)
    tag = str(getattr(d, "Laterality", "") or "").strip().upper()
    try:
        ipp = np.array(d.ImagePositionPatient, float); iop = np.array(d.ImageOrientationPatient, float)
        ps = np.array(d.PixelSpacing, float)
        c = ipp + iop[:3] * ps[1] * d.Columns / 2 + iop[3:] * ps[0] * d.Rows / 2
        return float(c[0]), tag
    except Exception:
        return np.nan, tag

SIDE = {}
for uid, g in tqdm(series.groupby("StudyInstanceUID"), desc="Sisi lutut"):
    xs, tags = [], []
    for s in g[g["Anatomical_Plane"].isin(["Coronal", "Axial"])]["SeriesInstanceUID"]:
        x, t = center_x(f"{COMP}/test_series/{uid}/{s}")
        if not np.isnan(x): xs.append(x)
        if t in ("L", "R"): tags.append(t)
    cx = float(np.median(xs)) if xs else np.nan
    if not np.isnan(cx) and abs(cx) > 20:
        SIDE[uid] = "L" if cx > 0 else "R"
    elif tags:
        SIDE[uid] = max(set(tags), key=tags.count)
    elif not np.isnan(cx):
        SIDE[uid] = "L" if cx > 0 else "R"
    else:
        SIDE[uid] = "R"
FLIP = {u: SIDE.get(u, "R") == "L" for u in uids}
print("Sisi lutut data uji:", pd.Series(SIDE).value_counts().to_dict())

Sisi lutut:   0%|          | 0/3 [00:00<?, ?it/s]

Sisi lutut data uji: {'R': 2, 'L': 1}


In [5]:
member_preds = {}
for name in USE:
    backbone, wfile, ver, lat = MODELS[name]
    ok = avail[ver]
    d0 = np.load(f"{VOL[ver]}/{ok[0]}.npz")
    n_views, depth = int(d0["mask"].shape[0]), int(d0["vol"].shape[1])
    flip_idx = {3: [1, 2], 5: [1, 2, 4]}[n_views] if lat else []
    ds = tb.KneeDS(ok, np.full((len(ok), len(LABELS)), np.nan, np.float32), VOL[ver], False,
                   FLIP if lat else None, flip_idx)
    bs = 1 if depth > 16 else 2
    dl = torch.utils.data.DataLoader(ds, batch_size=bs, shuffle=False, num_workers=2)

    net = tb.KneeNet(backbone, n_views=n_views, pretrained=False)
    net.load_state_dict(torch.load(find_file(wfile), map_location="cpu"))
    net = net.cuda().eval()
    if "vit" not in backbone:
        net = net.to(memory_format=torch.channels_last)
    p = tb.predict(net, dl)
    del net; gc.collect(); torch.cuda.empty_cache()

    p = pd.DataFrame(p, columns=LABELS); p.insert(0, "StudyInstanceUID", ok)
    member_preds[name] = test[["StudyInstanceUID"]].merge(p, on="StudyInstanceUID", how="left")[LABELS].values
    print(f"{name:28s} (data {ver}: {n_views} view x {depth} irisan, cermin {flip_idx}) selesai | "
          f"rata-rata prediksi {np.nanmean(member_preds[name]):.3f}")

effb0                        (data v1: 3 view x 16 irisan, cermin []) selesai | rata-rata prediksi 0.182
dinov2s_student_v1d          (data v1: 3 view x 16 irisan, cermin []) selesai | rata-rata prediksi 0.259
dinov2s_student_v1d_lat      (data v1: 3 view x 16 irisan, cermin [1, 2]) selesai | rata-rata prediksi 0.257
dinov2s_student_v1d_lat_s7   (data v1: 3 view x 16 irisan, cermin [1, 2]) selesai | rata-rata prediksi 0.267
dinov2s_student_v2           (data v2: 5 view x 16 irisan, cermin []) selesai | rata-rata prediksi 0.189
effb0_student_v3             (data v2: 5 view x 16 irisan, cermin []) selesai | rata-rata prediksi 0.265
dinov2s_student_d32_lat      (data v3: 5 view x 32 irisan, cermin [1, 2, 4]) selesai | rata-rata prediksi 0.238
effb0_student_d32_s7         (data v3: 5 view x 32 irisan, cermin []) selesai | rata-rata prediksi 0.194


In [6]:
p_ens = np.nanmean(np.stack([member_preds[n] for n in USE]), axis=0)
print("Anggota ensemble:", USE, "| bentuk:", p_ens.shape)

sub = pd.DataFrame(p_ens, columns=LABELS)
sub.insert(0, "StudyInstanceUID", uids)
n_missing = sub[LABELS].isna().any(axis=1).sum()
sub[LABELS] = sub[LABELS].fillna(0.5)

sample = pd.read_csv(f"{COMP}/sample_submission.csv")
sub = sub[sample.columns]
sub.to_csv("/kaggle/working/submission.csv", index=False)

print("Studi tanpa prediksi (diisi 0.5):", n_missing)
print("Shape submission:", sub.shape, "| shape sample:", sample.shape)
print(sub.head())

Anggota ensemble: ['effb0', 'dinov2s_student_v1d', 'dinov2s_student_v1d_lat', 'dinov2s_student_v1d_lat_s7', 'dinov2s_student_v2', 'effb0_student_v3', 'dinov2s_student_d32_lat', 'effb0_student_d32_s7'] | bentuk: (3, 12)
Studi tanpa prediksi (diisi 0.5): 0
Shape submission: (3, 13) | shape sample: (3, 13)
                                    StudyInstanceUID       ACL       MCL  \
0  1.2.826.0.1.3680043.8.498.10047035057544427318...  0.015510  0.070718   
1  1.2.826.0.1.3680043.8.498.10062861783145312629...  0.379695  0.099575   
2  1.2.826.0.1.3680043.8.498.10067514707072572280...  0.005588  0.035874   

   Medial Meniscus  Lateral Meniscus  Medial OA  Lateral OA     PF OA  \
0         0.419403          0.057220   0.250507    0.096774  0.253112   
1         0.360420          0.284047   0.292820    0.234205  0.343793   
2         0.419517          0.183448   0.384613    0.207277  0.697362   

   Effusion  Synovitis   Baker's  Contusion  Fracture  
0  0.493087   0.078642  0.706074   0.0945